In [ ]:
# ==========================================
# Celda 1: Importación de librerías
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import joblib

print("Librerías de modelado importadas correctamente.")

# ==========================================
# Celda 2: Carga de datos y variables base
# ==========================================
DATA_PATH = Path('data/results.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/results.csv')

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df = df.dropna(subset=['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral']).copy()

# Orden cronológico estricto
df = df.sort_values('date').reset_index(drop=True)

# Variable objetivo (Local, Empate, Visita)
df['resultado'] = np.select(
    [df['home_score'] > df['away_score'], df['home_score'] < df['away_score']],
    ['gana_local', 'gana_visita'],
    default='empate'
)

# Variables de tiempo
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# ==========================================
# Celda 3: Definición de features y Partición cronológica
# ==========================================
# EXCLUIMOS home_score y away_score para evitar data leakage
features = ['home_team', 'away_team', 'tournament', 'neutral', 'year', 'month']
X = df[features]
y = df['resultado']

# Partición cronológica: shuffle=False garantiza que el pasado entrena al futuro
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, shuffle=False)

train_start, train_end = df['date'].iloc[X_train.index[0]].date(), df['date'].iloc[X_train.index[-1]].date()
test_start, test_end = df['date'].iloc[X_test.index[0]].date(), df['date'].iloc[X_test.index[-1]].date()

print(f'Entrenamiento (Pasado): {X_train.shape[0]:,} partidos ({train_start} a {train_end})')
print(f'Prueba (Futuro): {X_test.shape[0]:,} partidos ({test_start} a {test_end})')

# ==========================================
# Celda 4: Creación del Pipeline y Entrenamiento
# ==========================================
# OneHotEncoding para variables de texto y Scaler para fechas
categorical_features = ['home_team', 'away_team', 'tournament', 'neutral']
numeric_features = ['year', 'month']

preprocesador = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

# Usamos Regresión Logística
modelo_futbol = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('clasificador', LogisticRegression(max_iter=2000, n_jobs=-1))
])

print("\nEntrenando el modelo...")
modelo_futbol.fit(X_train, y_train)
print("¡Modelo entrenado con éxito!")

# ==========================================
# Celda 5: Evaluación del Modelo
# ==========================================
y_pred = modelo_futbol.predict(X_test)

print("\n--- REPORTE DE CLASIFICACIÓN (TEST FUTURO) ---")
print(classification_report(y_test, y_pred))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred, 
    ax=ax, 
    cmap='Blues', 
    labels=['gana_local', 'empate', 'gana_visita']
)
plt.title('Matriz de Confusión - Regresión Logística (Fútbol)')
plt.grid(False)
plt.show()

# ==========================================
# Celda 6: Serialización y Prueba de Predicción
# ==========================================
ruta_modelo = 'modelo_futbol.pkl'
joblib.dump(modelo_futbol, ruta_modelo)
print(f'\n¡Modelo guardado en disco como "{ruta_modelo}"!')

# Prueba con un partido de exhibición
partido_ficticio = pd.DataFrame([{
    'home_team': 'Peru',
    'away_team': 'Denmark',
    'tournament': 'FIFA World Cup',
    'neutral': True,
    'year': 2026,
    'month': 6
}])

prediccion = modelo_futbol.predict(partido_ficticio)[0]
probabilidades = modelo_futbol.predict_proba(partido_ficticio)[0]
clases = modelo_futbol.classes_

print("\n--- PRUEBA DE PREDICCIÓN EN VIVO ---")
print(f"Encuentro: {partido_ficticio['home_team'].iloc[0]} vs {partido_ficticio['away_team'].iloc[0]}")
print(f"Predicción del modelo: {prediccion.upper()}")
print("Probabilidades calculadas:")
for clase, prob in zip(clases, probabilidades):
    print(f" - {clase}: {prob:.1%}")